In [ ]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [ ]:
def extract_text_details(text):
    """
    Identifies cited cases by searching for the 'v.' pattern.
    Commonly found in the judgment body or 'relied on' sections.
    """
    # Pattern looks for: Name v. Name (Year) Citation
    # Examples like: Gurupad Khandappa Magdum v. Hirabai Khandappa Magdum (1978) 3 SCC 383
    precedent_pattern = re.compile(r"([A-Z][\w\s\.\&]+ v\. [A-Z][\w\s\.\&]+(?:\s\(\d{4}\)\s\d+\s[A-Z\s]+(?:\d+)?))")
    
    # Extract and remove duplicates
    found = precedent_pattern.findall(text)
    return list(set([p.strip() for p in found]))

In [ ]:
# def extract_from_html(html_content):
#     """
#     Extracts structured fields from the raw_html found in the metadata JSONs.
#     """
#     data = {
#         "coram": None,
#         "decision_date": None,
#         "case_no": None,
#         "disposal_nature": None
#     }
    
#     if not html_content:
#         return data

#     # Regex patterns based on the SCR metadata format
#     patterns = {
#         "coram": r"Coram : (.*?)<br>",
#         "decision_date": r"Decision Date :</span><font color='green'> (.*?)</font>",
#         "case_no": r"Case No :</span><font color='green'> (.*?)</font>",
#         "disposal_nature": r"Disposal Nature :</span><font color='green'> (.*?)</font>"
#     }

#     for key, pattern in patterns.items():
#         match = re.search(pattern, html_content)
#         if match:
#             # Clean up HTML tags from the result
#             clean_val = re.sub(r'<.*?>', '', match.group(1)).strip()
#             data[key] = clean_val
            
#     return data

In [ ]:
def extract_from_html(html_content):
    data = {"coram": None, "decision_date": None, "case_no": None, "disposal_nature": None}
    if not html_content: return data
    
    patterns = {
        "coram": r"Coram : (.*?)<br>",
        "decision_date": r"Decision Date :</span><font color='green'> (.*?)</font>",
        "case_no": r"Case No :</span><font color='green'> (.*?)</font>",
        "disposal_nature": r"Disposal Nature :</span><font color='green'> (.*?)</font>"
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, html_content)
        if match:
            # Strips any residual HTML like <strong> or <sup>
            clean_val = re.sub(r'<.*?>', '', match.group(1)).strip()
            data[key] = clean_val
    return data

In [ ]:
def build_legal_parquet(text_folder, metadata_root, output_path):
    all_records = []
    
    # List all cleaned text files
    text_files = [f for f in os.listdir(text_folder) if f.endswith('.txt')]
    print(f"Found {len(text_files)} text files. Beginning merge...")

    for filename in tqdm(text_files):
        # 1. Load the Text
        file_path = os.path.join(text_folder, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            text_data = f.read()

        # 2. Derive Metadata ID and Year
        # Example: '2020_1_41_53_EN.txt' -> ID: '2020_1_41_53', Year: '2020'
        base_id = filename.replace('_EN.txt', '').replace('.txt', '')
        year = base_id.split('_')[0]
        
        # 3. Locate the Metadata JSON
        # Path format: D:\LPA_Part2\My_Datasets\SC_2020-2025\{year}\metadata\{id}.json
        json_filename = f"{base_id}.json"
        json_path = os.path.join(metadata_root, year, "metadata", json_filename)

        record = {
            "file_id": base_id,
            "year": year,
            "text": text_data,
            "coram": None,
            "decision_date": None,
            "case_no": None,
            "disposal_nature": None,
            "neutral_citation": None
        }

        # 4. Enrich with JSON data if it exists
        if os.path.exists(json_path):
            try:
                with open(json_path, 'r', encoding='utf-8') as jf:
                    meta_json = json.load(jf)
                    
                    html_data = extract_from_html(meta_json.get("raw_html", ""))
                    record.update(html_data)
                    record["neutral_citation"] = meta_json.get("nc_display")
            except Exception as e:
                print(f"Error processing metadata for {base_id}: {e}")

        all_records.append(record)

    # 5. Save to Parquet
    df = pd.DataFrame(all_records)
    df.to_parquet(output_path, engine='pyarrow', index=False)
    print(f"Successfully saved {len(df)} records to {output_path}")

In [ ]:
def build_final_parquet():
    # Paths based on your setup
    text_folder = r"D:\LPA_Part2\My_Datasets\Text_Datasets\texts_2020-2025"
    metadata_root = r"D:\LPA_Part2\My_Datasets\SC_2020-2025"
    output_path = r"D:\LPA_Part2\My_Datasets\Text_Datasets\SC_Legal_Kaggle_Ready.parquet"

    all_records = []
    text_files = [f for f in os.listdir(text_folder) if f.endswith('.txt')]

    for filename in tqdm(text_files, desc="Building Dataset"):
        # 1. Load Full Text
        with open(os.path.join(text_folder, filename), 'r', encoding='utf-8') as f:
            content = f.read()

        # 2. Match IDs (Strip _EN)
        base_id = filename.replace('_EN.txt', '').replace('.txt', '')
        year = base_id.split('_')[0]
        
        # 3. Path for Metadata: .../2023/metadata/2023_1_1_10.json
        json_path = os.path.join(metadata_root, year, "metadata", f"{base_id}.json")

        # 4. Create Record
        record = {
            "file_id": base_id,
            "year": year,
            "full_text": content,
            "precedents": extract_precedents(content)
        }

        # 5. Enrich with Metadata
        if os.path.exists(json_path):
            with open(json_path, 'r', encoding='utf-8') as jf:
                meta_json = json.load(jf)
                record.update(extract_from_html(meta_json.get("raw_html", "")))
                record["neutral_citation"] = meta_json.get("nc_display")

        all_records.append(record)

    # 6. Save for Kaggle
    df = pd.DataFrame(all_records)
    df.to_parquet(output_path, engine='pyarrow', index=False, compression='snappy')
    print(f"\nSaved {len(df)} cases to {output_path}")

In [ ]:
start_year = str(input("Enter start year: "))
end_year = str(input("Enter end year: "))

if __name__ == "__main__":
    TEXT_DIR = f"../../My_Datasets/Text_Datasets/texts_{start_year}-{end_year}"
    META_ROOT = f"../../My_Datasets/SC_{start_year}-{end_year}"
    OUTPUT = f"../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_{start_year}-{end_year}.parquet"

    build_legal_parquet(TEXT_DIR, META_ROOT, OUTPUT)